In [61]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/checking-logs.sqlite")

In [ ]:
query = """
WITH ABdataCTE AS (
    SELECT 
        g.uid,
        (strftime('%s', g.first_commit_ts) - d.deadlines) / 3600 AS diff,
        IIF(strftime('%s', g.first_commit_ts) < strftime('%s', g.first_view_ts), 'before', 'after') AS period
    FROM {group} AS g
    JOIN deadlines AS d ON g.labname = d.labs
    WHERE g.labname != 'project1'
),
TargetUsersCTE AS (
    SELECT uid 
    FROM ABdataCTE 
    GROUP BY uid 
    HAVING COUNT(DISTINCT period) = 2
),
UserAvgCTE AS (
    SELECT 
        uid,
        period,
        AVG(diff) AS user_avg_diff
    FROM ABdataCTE
    WHERE uid IN (SELECT uid FROM TargetUsersCTE)
    GROUP BY uid, period
)
SELECT 
    period AS time,
    AVG(user_avg_diff) AS avg_diff
FROM UserAvgCTE
GROUP BY period;
"""

In [64]:
test_results = pd.io.sql.read_sql(query.format(group="test"), conn)
test_results

,time,avg_diff
0,after,-99.523810
1,before,-66.047619


In [65]:
control_results = pd.io.sql.read_sql(query.format(group="control"), conn)
control_results

,time,avg_diff
0,after,-99.322222
1,before,-98.033333


In [66]:
conn.close()

## Дай ответ: подтвердилась ли гипотеза - влияет ли страница на поведение студентов?

Да, страница ленты новостей повлияла на поведение студентов. До просмотра ленты студенты сдавали работы в среднем за 66 часов до дедлайна, а после начала чтения ленты стали сдавать за 99.5 часов до дедлайна. 
В контрольную группу в ex02 мы выделяли пользователей, которые не смотрели ленту - (first_view_ts.isna()) и  заполнили эти пропуски средним значением из группы test. Суть A/B-теста — смотреть не на итоговую цифру, а на прогресс внутри каждой группы, то есть получается, что условно для лентяеев (тестовая) лента сработала, увеличив их скорость на 33,5 часа, а у более продуктивных итак результат был довольно высоким (разница в 1:20).